# Silver Transform
Test the Silver layer transformation independently.
Note: You should have run bronze_ingestion first so there is pending Bronze data.

In [1]:
from datetime import datetime, timezone
import polars as pl
from electricity_maps.layers.silver import transform_silver_flows, transform_silver_mix
from electricity_maps.utils.state import PipelineState
from electricity_maps.utils.storage import delta_storage_options
from electricity_maps.config import get_settings

In [2]:
get_settings.cache_clear()
settings = get_settings()
state = PipelineState(settings.data_dir)

now = datetime.now(timezone.utc)
silver_ts = int(now.timestamp())

In [3]:
ts_list = state.pickup_ready("bronze")
if not ts_list:
    print("No pending bronze data found. Run 02_bronze_ingestion first.")
else:
    print(f"Picked up Bronze batches: {ts_list}")
    
    transform_silver_mix(ts_list)
    transform_silver_flows(ts_list)
    
    # Mark state
    state.mark_complete("bronze", ts_list)
    state.init_layer("silver", silver_ts, now)
    state.mark_ready("silver", silver_ts, now, len(ts_list))
    
    print("Silver transformation completed!")

2026-04-25T08:39:01.247913Z [info     ] el_state: no ready rows to pick up [electricity_maps.utils.state] process=bronze
No pending bronze data found. Run 02_bronze_ingestion first.


In [4]:
# Let's inspect the Silver Delta tables
print("Silver Mix Table:")
try:
    path = str(settings.silver_dir / "electricity_mix")
    df_mix = pl.read_delta(path, storage_options=delta_storage_options(path))
    display(df_mix.head())
except Exception as e:
    print("Error reading silver mix:", e)

Silver Mix Table:


zone,datetime,updated_at,is_estimated,estimation_method,nuclear_mw,geothermal_mw,biomass_mw,coal_mw,wind_mw,solar_mw,hydro_mw,gas_mw,oil_mw,unknown_mw,hydro_storage_charge_mw,hydro_storage_discharge_mw,battery_storage_charge_mw,battery_storage_discharge_mw,flow_exports_mw,flow_imports_mw,year,month,day
str,"datetime[μs, UTC]","datetime[μs, UTC]",bool,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i32,i32,i32
"""FR""",2026-04-25 06:00:00 UTC,2026-04-25 08:09:01.663 UTC,true,"""SANDBOX_MODE_DATA""",34990.256,null,1036.715,0.0,1591.403,2916.188,6610.653,383.766,35.796,null,null,377.163,null,29.501,17562.0,0.0,2026,4,25
"""FR""",2026-04-25 01:00:00 UTC,2026-04-25 03:08:12.543 UTC,true,"""SANDBOX_MODE_DATA""",50500.296,null,1355.03,0.0,4131.045,0.0,4770.328,415.109,33.764,null,null,885.845,17.495,null,20768.0,0.0,2026,4,25
"""FR""",2026-04-25 03:00:00 UTC,2026-04-25 05:23:17.793 UTC,true,"""SANDBOX_MODE_DATA""",45232.031,null,1318.573,0.0,2881.676,0.0,4281.228,454.126,40.898,null,null,542.241,12.532,null,17494.0,0.0,2026,4,25
"""FR""",2026-04-25 02:00:00 UTC,2026-04-25 04:23:38.601 UTC,true,"""SANDBOX_MODE_DATA""",49023.194,null,1050.156,0.0,4111.303,0.0,3765.013,276.942,37.309,null,null,356.625,0.617,null,21599.0,0.0,2026,4,25
"""FR""",2026-04-25 07:00:00 UTC,2026-04-25 08:08:49.055 UTC,true,"""SANDBOX_MODE_DATA""",49960.602,null,1405.958,0.0,1209.611,3035.377,4654.037,283.434,42.038,null,null,437.291,null,27.967,11295.0,0.0,2026,4,25


In [5]:
print("\nSilver Flows Table:")
try:
    path = str(settings.silver_dir / "electricity_flows")
    df_flows = pl.read_delta(path, storage_options=delta_storage_options(path))
    display(df_flows.head())
except Exception as e:
    print("Error reading silver flows:", e)


Silver Flows Table:


zone,datetime,updated_at,neighbor_zone,direction,power_mw,year,month,day
str,"datetime[μs, UTC]","datetime[μs, UTC]",str,str,f64,i32,i32,i32
"""FR""",2026-04-25 03:00:00 UTC,2026-04-25 05:23:17.793 UTC,"""GB""","""export""",2247.0,2026,4,25
"""FR""",2026-04-25 01:00:00 UTC,2026-04-25 03:08:12.543 UTC,"""CH""","""import""",1815.0,2026,4,25
"""FR""",2026-04-25 05:00:00 UTC,2026-04-25 07:10:34.382 UTC,"""LU""","""export""",144.0,2026,4,25
"""FR""",2026-04-25 03:00:00 UTC,2026-04-25 05:23:17.793 UTC,"""LU""","""export""",248.0,2026,4,25
"""FR""",2026-04-25 04:00:00 UTC,2026-04-25 06:24:03.931 UTC,"""ES""","""import""",2555.0,2026,4,25
